# Lab6.2 — Pipeline hardware streaming: arhitectură și flux de date

**Întrebarea centrală:** ce se schimbă în arhitectură atunci când legăm direct Resize de Sobel prin AXI4-Stream, fără să stocăm imaginea intermediară în DDR?

Acesta este notebook-ul cu accent pe **proiectarea sistemului hardware**. Vom implementa pe ARM un model funcțional Sobel cu trei linii, vom analiza protocolul AXI4-Stream și vom calcula traficul DDR estimat. Acestea sunt **modele software și calcule**, nu măsurători FPGA.

**Arhitectura-țintă:** DDR → DMA MM2S → Resize (PL) → Grayscale (PL) → Sobel (PL) → DMA S2MM → DDR.

Pentru execuția acestei arhitecturi în PL avem nevoie de un overlay nou, cu fișierele `.bit` și `.hwh` validate.

## 1. Arhitectură secvențială versus pipeline streaming

**Două acceleratoare independente:** DDR → DMA → Resize → DMA → DDR intermediar → DMA → Sobel → DMA → DDR rezultat.

**Pipeline în flux:** DDR → DMA → Resize → Grayscale → Sobel → DMA → DDR rezultat.

A doua arhitectură elimină **scrierea și citirea imaginii intermediare** din DDR. Totuși, Sobel necesită acces local la vecinătatea 3×3, deci folosește buffere de linii (în general BRAM și registre), fără a stoca întregul cadru în DDR.

**Cerință:** desenați cele două diagrame și marcați magistralele, lățimea pixelilor și punctele unde poate apărea backpressure. Explicați diferența dintre buffer de linii și buffer de cadru.

## 2. Model funcțional Sobel cu trei linii

Pornim de la aceeași imagine ca în Lab6.1. Modelăm Sobel la nivel de **linii succesive**, păstrând linia precedentă, curentă și următoare.

Pentru tratarea marginilor vom folosi completare cu zerouri (`BORDER_CONSTANT`). Referința OpenCV trebuie să utilizeze aceeași politică. Altfel, diferențele la margini nu reprezintă o eroare de implementare.

**Important:** codul Python de mai jos demonstrează algoritmul; nu reprezintă simularea exactă pe cicluri a unui IP FPGA. Matricea rezultat este păstrată pentru verificare și vizualizare.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from lab6_common import load_test_image, sobel_software, compare_images

bgr = load_test_image()
TARGET = (320, 240)
small = cv2.resize(bgr, TARGET, interpolation=cv2.INTER_LINEAR)
gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)

def sobel_three_rows(gray_image):
    """Model Sobel cu trei linii, cu zerouri la marginile imaginii."""
    if gray_image.ndim != 2 or gray_image.dtype != np.uint8:
        raise TypeError("Se asteapta o imagine grayscale uint8.")
    h, w = gray_image.shape
    zero = np.zeros(w, dtype=np.int32)
    previous = zero
    current = gray_image[0].astype(np.int32)
    following = gray_image[1].astype(np.int32) if h > 1 else zero
    output = np.empty((h, w), dtype=np.uint8)
    for y in range(h):
        p = np.pad(previous, (1, 1), mode="constant")
        q = np.pad(current, (1, 1), mode="constant")
        n = np.pad(following, (1, 1), mode="constant")
        gx = (-p[:-2] + p[2:] - 2*q[:-2] + 2*q[2:]
              -n[:-2] + n[2:])
        gy = (-p[:-2] - 2*p[1:-1] - p[2:]
              +n[:-2] + 2*n[1:-1] + n[2:])
        magnitude = np.sqrt(gx.astype(np.float32)**2 +
                            gy.astype(np.float32)**2)
        output[y] = np.clip(np.rint(magnitude), 0, 255).astype(np.uint8)
        previous, current = current, following
        following = (gray_image[y+2].astype(np.int32)
                     if y+2 < h else zero)
    return output

reference = sobel_software(gray, border_type=cv2.BORDER_CONSTANT)
stream_model = sobel_three_rows(gray)
print("OpenCV versus model cu trei linii:", compare_images(reference, stream_model))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, pic, title in zip(axes,
        [reference, stream_model,
         np.abs(reference.astype(np.int16)-stream_model.astype(np.int16))],
        ["Referinta OpenCV", "Model cu trei linii", "Diferenta absoluta"]):
    ax.imshow(pic, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()

### Întrebări de validare

1. De câte linii ale imaginii are nevoie efectiv o fereastră 3×3?
2. Cum se modifică memoria necesară bufferelor când dublăm lățimea imaginii?
3. Ce efect are tratarea marginilor asupra pixelilor de pe contur?
4. De ce pot apărea mici diferențe numerice față de OpenCV chiar folosind aceiași coeficienți Sobel?

**Experiment suplimentar:** modificați tratarea marginilor și configurați corespunzător referința OpenCV. Justificați diferențele.

## 3. AXI4-Stream: TVALID, TREADY și delimitarea datelor

Un transfer AXI4-Stream se produce numai când **TVALID = 1 și TREADY = 1**, la frontul activ de ceas. Dacă receptorul nu este pregătit, emițătorul trebuie să păstreze stabile datele și semnalele laterale. Această mecanică permite propagarea backpressure-ului între etapele pipeline-ului.

Simulăm un pachet de date pentru care receptorul produce periodic opriri. În exemplul didactic, **TLAST** marchează sfârșitul pachetului.

**Atenție:** în AXI4-Stream Video, TLAST marchează în mod obișnuit sfârșitul unei linii, iar TUSER indică începutul cadrului. Formatul efectiv și adaptarea pentru DMA trebuie verificate pentru fiecare IP folosit.

In [ ]:
def simulate_stream_packet(n_pixels, stall_period=5):
    queued = list(range(n_pixels))
    accepted, trace = [], []
    pending = None
    clock = 0
    while len(accepted) < n_pixels:
        if pending is None:
            ix = len(accepted)
            pending = (queued[ix], ix == n_pixels-1)
        valid = True
        ready = clock % stall_period != stall_period-1
        data, last = pending
        transferred = bool(valid and ready)
        if len(trace) < 18:
            trace.append((clock, int(valid), int(ready),
                          data, int(last), int(transferred)))
        if transferred:
            accepted.append(data)
            pending = None
        clock += 1
    assert accepted == queued
    return clock, trace

cycles, trace = simulate_stream_packet(32, stall_period=5)
print("CICLU VALID READY DATE LAST ACCEPTAT")
for row in trace:
    print("%5d %5d %5d %4d %4d %8d" % row)
print("32 transferuri in", cycles, "cicluri; medie:",
      round(32/cycles, 3), "beats/clock")

## 4. Estimarea traficului DDR

**Ipoteze:** imaginea inițială este BGR uint8 (3 octeți/pixel); rezultatul Resize este tot BGR uint8; rezultatul final Sobel este grayscale uint8 (1 octet/pixel).

Ignorăm pentru acest model overhead-ul AXI, alinierea transferurilor, efectele cache și celelalte copieri efectuate de software.

Pentru **două acceleratoare separate**: datele citite la intrare + **2 × dimensiunea imaginii intermediare** (scriere și citire) + rezultatul final.

Pentru **pipeline-ul în flux**: datele citite la intrare + rezultatul final.

Diferența teoretică este de două transferuri complete ale imaginii intermediare. Aceste valori reprezintă **volumul minim de date utile**, nu tranzacții DDR măsurate.

In [ ]:
source_h, source_w, _ = bgr.shape
sizes = [(160, 120), (320, 240), (480, 320)]
print("%-12s %15s %15s %13s" %
      ("Iesire", "Separat MiB", "Streaming MiB", "Reducere [%]"))
traffic = []
for w, h in sizes:
    source_bytes = source_w * source_h * 3
    intermediate_bytes = w * h * 3
    result_bytes = w * h  # grayscale Sobel result
    separate = source_bytes + 2*intermediate_bytes + result_bytes
    fused = source_bytes + result_bytes
    saving_percent = 100*(separate-fused)/separate
    traffic.append((w, h, separate, fused))
    print("%-12s %15.3f %15.3f %13.2f" %
          ("%dx%d" % (w, h), separate/2**20,
           fused/2**20, saving_percent))

labels = ["%dx%d" % (w, h) for w, h, _, _ in traffic]
x = np.arange(len(labels))
plt.figure(figsize=(8, 4))
plt.bar(x-0.18, [a/2**20 for _, _, a, _ in traffic],
        width=0.36, label="Acceleratoare separate")
plt.bar(x+0.18, [b/2**20 for _, _, _, b in traffic],
        width=0.36, label="Pipeline AXI4-Stream")
plt.xticks(x, labels)
plt.ylabel("Date utile DDR minime/cadru [MiB]")
plt.title("Model analitic — nu masuratori DDR")
plt.legend(); plt.tight_layout(); plt.show()

## 5. De la model la overlay-ul PYNQ-Z2

Pentru implementarea în Vivado/Vitis 2022.2 proiectăm:

1. AXI DMA cu canale MM2S și S2MM conectate la DDR.
2. Un accelerator Resize cu intrare/ieșire AXI4-Stream și control AXI-Lite.
3. Conversie BGR/RGB → grayscale și Sobel 3×3 cu buffere de linii.
4. Legături streaming directe între IP-uri, cu semnalele de handshake și delimitare compatibile.
5. Un flux de ieșire adaptat la DMA și o pereche `.bit` + `.hwh` corespunzătoare.

**Verificări obligatorii:** ordinea canalelor BGR/RGB, lățimea `TDATA`, numărul de octeți, `TLAST`, `TUSER`, `TKEEP`, politica marginilor, terminarea transferului DMA.

**Stadiul actual:** repository-ul include modelul software și specificația de implementare, dar **nu** include încă un bitstream verificat pentru Resize → Sobel integral în PL.

## 6. Sarcină integratoare — document tehnic

Elaborați un document de 1–2 pagini care să includă:

- diagrama variantelor secvențială și streaming, cu DDR, DMA, AXI-Lite și AXI4-Stream;
- validarea modelului Sobel cu trei linii și diferențele numerice obținute;
- interpretarea unei opriri `TREADY=0` din tabelul de semnale;
- calculul traficului DDR pentru încă o rezoluție aleasă;
- un plan de validare a unui overlay real, incluzând `ip_dict`, controlul DMA, corectitudinea și performanța.

**Legătura cu Lab6.3:** reducerea traficului DDR reprezintă un avantaj arhitectural potențial; impactul real asupra FPS-ului aplicației video se stabilește numai prin măsurători.